## *如何使用 operations_definition：*

**目标：** 为使用 VTA 模拟器，需要将指令生成为二进制文件供模拟器读取。以下程序利用 *data_definition/* 中获取的数据，为 VTA 需要执行的每个操作（load、GeMM、ReLU、ALU、store、reset 等）生成相应指令。

**生成指令：**
以 LeNet-5 第一卷积层（GeMM）为例，之后接 ReLU 和平均池化（目的是缩小 GeMM 后输出矩阵的尺寸）：
当前，我们有 16x16 的 INP 块矩阵 *Ai* 和 16x16 的 WGT 矩阵 *Bi*。要执行 **GeMM**，*Ai* 需要被分割为 (16x1) 的横向向量，得到由 (16x1) 横向向量组成的块输出矩阵 *ACCi*，再将矩阵重新组合为 (16x16) 块。
然后对 *ACCi* 应用 **ReLU**。得到的块矩阵随后进行平均池化，由 **2 次 ADD** 和 **1 次 SHR**（数据除以 4）组成。

对于各操作（reset、GEMM、ALU），需要填充 UOP 缓冲区（`VTAUop` 架构，参见 `structures_insn_uop.py`），以确定数据的读/写索引。
指令缓冲区的字段（GEMM 指令用 `VTAGemInsn`，ALU 指令用 `VTAAluInsn`，存储/加载指令用 `VTAMemInsn`）也需根据输入张量和滤波器的维度进行配置。

- **LOAD（128 位）：** 从 DRAM 中提取数据并临时存储在 SRAM（VTA 内部的本地内存）中，以进行运算
- **ALU（128 位）：** ReLU 激活（对矩阵中每个值 x => max(0, x)）、ADD 1 & 2，然后 SHR（对部分数据求平均）
- **GEMM（128 位）：** A 与 B 的转置矩阵乘法
- **STORE（128 位）：** 所有操作完成后，将 SRAM 中的数据复制到 DRAM 中的目标位置

*以程序 `insn_lenet5_layer1.py` 为例，演示如何将数据生成为二进制文件：*

### 配置（CONFIGURATION）

初始化缓冲区并从 `structures_insn_uop.py` 导入必要的数据结构。
缓冲区用于存储以下数据：
- **UOP 缓冲区**：存储每条指令的索引。UOP 缓冲区有三个字段：ACC、INP、WGT，索引为操作执行的起始位置。
- **指令缓冲区**：描述每条指令的内存存储。缓冲区的结构根据用途不同而异，分为内存交互类型和操作定义类型。

In [ ]:
"""配置"""

# 导入包
# --------------
%pip install numpy
import numpy as np
import sys

# 父目录
sys.path.append('../src/compiler/vta_compiler/operations_definition')
sys.path.append('../src/compiler/vta_compiler/data_definition')
import matrix_generator as MG
import matrix_split as MS
import structures_insn_uop

# UOP 定义
# --------------
# 初始化空 UOP 缓冲区
uop_buffer = []

# 指令定义
# ----------------------
# 初始化空指令缓冲区
insn_buffer = []

# 输入数据
# --------------
block_size = 16

# A 矩阵尺寸
A_row = 784
A_col = 25

block_input_matrix, _ = MS.matrix_splitting(MG.matrix_padding(MG.matrix_creation(n_row=A_row, n_col=A_col, isInitRandom=True, random_bound=4)), block_size, isWeight=False, isSquare=True)

# B 矩阵尺寸
B_row = A_col # 矩阵乘法要求
B_col = 6

block_weight_matrix, _ = MS.matrix_splitting(MG.matrix_padding(MG.matrix_creation(n_row=B_row, n_col=B_col, isInitRandom=True, random_bound=4)), block_size, isWeight=True, isSquare=True)

#### 从 DRAM 加载数据（LOAD DATA FROM DRAM）

第一个缓冲区用于在 GEMM 之前重置 ACC。
各字段表示数据读写的初始索引：
- `dst_idx`：数据写入位置（ACC）
- `src_idx`：数据读取位置（ACC 或 INP）
- `wgt_idx`：WGT 中的数据读取位置（仅用于 GEMM）

In [ ]:
if (len(uop_buffer) < 1):
    uop_buffer.append(structures_insn_uop.VTAUop( # UOP 0 - 重置
        dst_idx=0, 
        src_idx=0,
        wgt_idx=0
    ))

###### *内存缓冲区（MEMORY BUFFERS）*

以下缓冲区模拟 LOAD 模块与 COMPUTE 模块之间的内存交互。

各字段含义如下：
- `opcode`：指示缓冲区所定义的操作类型
- DEP FLAG（依赖标志）：`pop_dep` 请求发送下一条指令的许可，`push_dep` 给予许可。这是为了确保数据处理时不发生重叠。
- `buffer_id`：指示正在读取的数据类型
- `sram_base`、`dram_base`：LOAD 操作时，sram 是本地内存地址，数据从 dram 复制到 sram
- `y_size`、`x_size` 等：数据占用的内存大小

In [ ]:
if (len(insn_buffer) < 1):
    
# 加载 RESET UOP

    insn_buffer.append(structures_insn_uop.VTAMemInsn( # I0: LOAD UOP
        opcode=0, # 0-LOAD（加载）, 1-STORE（存储）, 3-FINISH（完成）
        # 依赖标志
        pop_prev_dep=0,
        pop_next_dep=0,
        push_prev_dep=0,
        push_next_dep=0,
        # 内存交互
        buffer_id=0, # 0-UOP, 1-WGT, 2-INP, 3-ACC, 4-OUT, 5-ACC8bit
        sram_base=0x0000,
        dram_base=0x00001000,
        unused=0, # 未使用
        # 数据操作
        y_size=1,
        x_size=1,
        x_stride=1,
        y_pad_top=0,
        y_pad_bottom=0,
        x_pad_left=0,
        x_pad_right=0
    ))

# RESET 时清空 ACC 矩阵

    insn_buffer.append(structures_insn_uop.VTAGemInsn( # I1: GEMM RESET
        opcode=2, # 2-GEMM
        # 依赖标志
        pop_prev_dep=0,
        pop_next_dep=0,
        push_prev_dep=1, # 发送就绪信号给 LOAD
        push_next_dep=0,
        # Operations
        reset=1, # 0-否, 1-重置
        uop_bgn=0, # UOP 0
        uop_end=1,
        loop_out=49, # Number of (16 x 16) blocks in ACC
        loop_in=16,  # Block size
        # 未使用
        unused=0, # 未使用
        # Index factors
        dst_factor_out=16, # Block size
        dst_factor_in=1,
        src_factor_out=0,
        src_factor_in=0,
        wgt_factor_out=0,
        wgt_factor_in=0
    ))
    
# 加载 INP

    insn_buffer.append(structures_insn_uop.VTAMemInsn( # I2: LOAD INP
        opcode=0, # 0-LOAD（加载）, 1-STORE（存储）, 3-FINISH（完成）
        # 依赖标志
        pop_prev_dep=0,
        pop_next_dep=1, # 确认 COMPUTE 就绪信号
        push_prev_dep=0,
        push_next_dep=0,
        # 内存交互
        buffer_id=2, # 0-UOP, 1-WGT, 2-INP, 3-ACC, 4-OUT, 5-ACC8bit
        sram_base=0x0000,
        dram_base=0x00000100,
        unused=0, # 未使用
        # 数据操作
        y_size=1,
        x_size=1568, # 加载 98*16 个 INP
        x_stride=1568,
        y_pad_top=0,
        y_pad_bottom=0,
        x_pad_left=0,
        x_pad_right=0
    ))
    
# 加载 WGT

    insn_buffer.append(structures_insn_uop.VTAMemInsn( # I3: LOAD WGT
        opcode=0, # 0-LOAD（加载）, 1-STORE（存储）, 3-FINISH（完成）
        # 依赖标志
        pop_prev_dep=0,
        pop_next_dep=0,
        push_prev_dep=0,
        push_next_dep=1, # Ready signal to COMPUTE
        # 内存交互
        buffer_id=1, # 0-UOP, 1-WGT, 2-INP, 3-ACC, 4-OUT, 5-ACC8bit
        sram_base=0x0000,
        dram_base=0x00000020,
        unused=0, # 未使用
        # 数据操作
        y_size=1,
        x_size=2, # 加载 2 个 WGT
        x_stride=2,
        y_pad_top=0,
        y_pad_bottom=0,
        x_pad_left=0,
        x_pad_right=0
    ))
    
# 为 GEMM 和平均池化操作加载 UOP

    insn_buffer.append(structures_insn_uop.VTAMemInsn( # I4: LOAD UOP
        opcode=0, # 0-LOAD（加载）, 1-STORE（存储）, 3-FINISH（完成）
        # 依赖标志
        pop_prev_dep=1, # 确认 LOAD 就绪信号
        pop_next_dep=0, 
        push_prev_dep=0,
        push_next_dep=0,
        # 内存交互
        buffer_id=0, # 0-UOP, 1-WGT, 2-INP, 3-ACC, 4-OUT, 5-ACC8bit
        sram_base=0x0001,
        dram_base=0x00001001,
        unused=0, # 未使用
        # 数据操作
        y_size=1,
        x_size=6, # 加载 6 个 UOP（2 GeMM + 1 ReLU + 3 Pool）
        x_stride=6,
        y_pad_top=0,
        y_pad_bottom=0,
        x_pad_left=0,
        x_pad_right=0
    ))

## GEMM

广义矩阵乘法（Generalized Matrix Multiplication）操作。不采用分块乘法，而是将所有 INP 块堆叠成一个 (1568x16) 矩阵并向量化。由于有两个 WGT 块矩阵（16x16），最终得到一个 (784x16) 的 ACC 矩阵。

VTA 执行的操作如下：INP 的每个向量与 WGT 的转置逐行相乘，结果存入 ACC。

#### INP 和 ACC 的向量化（VECTORIZATION OF INP AND ACC）

*硬件要求：* 对于 Compute 操作，VTA 要求 WGT 为 (`block_size=16` x `block_size=16`) 的矩阵，其余数据为 (`block_size=16` x 1) 的向量。

(`block_size=16` x `block_size=16`) 的 INP 和 ACC 矩阵需要分割为向量：后续操作均在向量上进行。

以下是 VTA 执行 GEMM 操作的示例：

In [ ]:
# ----------------------
# 将 (16x16) 的 INP 块矩阵分割为 (16x1) 向量

# 输入矩阵 INP
print("First vector of first block of INP matrix (", np.shape(block_input_matrix[0][0])[0], " x ", 1, ")")
print(block_input_matrix[0][0], "A@0")

# 权重矩阵 WGT
print("x \nFirst block of WGT matrix (", block_weight_matrix[0].shape[0], " x ", block_weight_matrix[0].shape[1], ")")
print(block_weight_matrix[0], "B@0")

# 输出矩阵 ACC
C_0 = np.zeros((1, block_size))
print("= \nFirst vector of first block of ACC [to be filled] (", block_size, " x ", 1, ")")
print(C_0, "C@0")

##### *操作缓冲区（OPERATION BUFFERS）*

以下缓冲区给出每个操作的结构。
各字段含义如下：
- `opcode`：指示缓冲区所定义的操作类型
- DEP FLAG（依赖标志）：`pop_dep` 请求发送下一条指令的许可，`push_dep` 给予许可。这是为了确保数据处理时不发生重叠。
- `reset`：若为 yes，则删除已存储的数据
- `uop_bgn`、`uop_end`：指令所需的 UOP 索引范围
- `loop_out`：外部循环次数，即操作的迭代次数
- `loop_in`：内部循环次数
- `dst_factor_out`、`dst_factor_in`：分别为每次 `loop_out`、`loop_in` 后 ACC 中索引的增量
- `src_factor_out`、`src_factor_in`：同上，但针对 INP
- `wgt_factor_out`、`wgt_factor_in`：同上，但针对 WGT

以下缓冲区对应的乘法操作为：

>*A0 * B0 +=> C0*

>*A1 * B1 +=> C0*

>*A2 * B0 +=> C1*

>*A3 * B1 +=> C1*

>*以此类推...*

In [ ]:
"""GEMM（广义矩阵乘法）"""

# 使用向量化的 A 和 B 生成 GeMM 指令

# ----------------------
# 定义 GEMM UOP 缓冲区

if (len(uop_buffer) < 1 + 1):
    uop_buffer.append(structures_insn_uop.VTAUop( # UOP 1 - GEMM 0
        dst_idx=0, 
        src_idx=0,
        wgt_idx=0
    ))

if (len(uop_buffer) < 2 + 1):
    uop_buffer.append(structures_insn_uop.VTAUop( # UOP 2 - GEMM 1
        dst_idx=0, 
        src_idx=16,
        wgt_idx=1
    ))

# ----------------------
# 定义 GEMM 指令缓冲区

index_insn = 5 # 指令索引

if (len(insn_buffer) < index_insn + 1):
    insn_buffer.append(structures_insn_uop.VTAGemInsn( # I5: GEMM
        opcode=2, # 2-GEMM
        # 依赖标志
        pop_prev_dep=0,
        pop_next_dep=0,
        push_prev_dep=0,
        push_next_dep=0, 
        # Operations
        reset=0, # 0-否, 1-重置
        uop_bgn=1, # UOP 1 + UOP 2
        uop_end=3,
        loop_out=49,
        loop_in=16,
        # 未使用
        unused=0, # 未使用
        # 索引因子
        dst_factor_out=16,
        dst_factor_in=1,
        src_factor_out=32,
        src_factor_in=1,
        wgt_factor_out=0,
        wgt_factor_in=0
    ))

# ----------------------
# 打印缓冲区
 
# 打印 UOP 缓冲区
def print_uop_buffer(OP, uop_bgn, uop_end) :
    print(OP, "UOP BUFFER\nACC  INP  WGT\n")
    for i in range(uop_bgn, uop_end):
        print(uop_buffer[i].dst_idx, "  ", uop_buffer[i].src_idx, "  ", uop_buffer[i].wgt_idx, "\n")

# 打印 ALU 指令缓冲区      
def print_insn_buffer_ALU(n_insn, OP):
    print(OP, "INSTRUCTIONS\nLP_OUT  LP_IN  DST_OUT  DST_IN  SRC_OUT  SRC_IN  OPCODE  IMM\n")
    print(insn_buffer[n_insn].loop_out, "     ", insn_buffer[n_insn].loop_in, "     ", insn_buffer[n_insn].dst_factor_out, "     ", insn_buffer[n_insn].dst_factor_in, "     ", 
          insn_buffer[n_insn].src_factor_out, "     ", 
          insn_buffer[n_insn].src_factor_in, "     ", insn_buffer[n_insn].opcode, "    ", insn_buffer[n_insn].imm)
    
# ----------------------
# 定义 GEMM 操作

def GEMM(A, B):
#    assert(A.shape[1] == B.shape[0])
    A = np.array(A)
    B = np.array(B)
    return A @ B

# ----------------------
# GEMM 伪代码

def insn_GEMM(ACC, WGT, INP):
    for i0 in range(insn_buffer[index_insn].loop_in):
        for i1 in range(insn_buffer[index_insn].loop_out):
            for uop_index in range(insn_buffer[index_insn].uop_bgn, insn_buffer[index_insn].uop_end):
                X, Y, Z = uop_buffer[uop_index].dst_idx, uop_buffer[uop_index].src_idx, uop_buffer[uop_index].wgt_idx
                dst_idx = i0 * insn_buffer[index_insn].dst_factor_in + i1 * insn_buffer[index_insn].dst_factor_out + X # ACC 中的索引
                inp_idx = i0 * insn_buffer[index_insn].src_factor_in + i1 * insn_buffer[index_insn].src_factor_out + Y # INP 中的索引
                wgt_idx = i0 * insn_buffer[index_insn].wgt_factor_in + i1 * insn_buffer[index_insn].wgt_factor_out + Z # WGT 中的索引
                ACC[dst_idx] += GEMM(INP[inp_idx], WGT[wgt_idx])                                                       # 将 GEMM(A, B) 的结果存入 ACC
    return ACC

# ----------------------
# 打印数据
# ----------------------

# 打印 GEMM UOP 缓冲区
print_uop_buffer("GEMM", insn_buffer[index_insn].uop_bgn, insn_buffer[index_insn].uop_end)

# 打印 GEMM 指令缓冲区 
print("GEMM INSTRUCTIONS\nLP_OUT  LP_IN  DST_OUT  DST_IN  SRC_OUT  SRC_IN  WGT_OUT  WGT_IN\n")
print(insn_buffer[index_insn].loop_out, "     ", insn_buffer[index_insn].loop_in, "     ", insn_buffer[index_insn].dst_factor_out, "     ", insn_buffer[index_insn].dst_factor_in, "     ", 
        insn_buffer[index_insn].src_factor_out, "     ", 
        insn_buffer[index_insn].src_factor_in, "     ", insn_buffer[index_insn].wgt_factor_out, "     ", insn_buffer[index_insn].wgt_factor_in, "\n")

# 打印输出矩阵
INP_stack = np.vstack(block_input_matrix)       # 将 A 的 98 个 (16x16) 块堆叠
ACC = np.zeros((A_row, block_size))             # 用零初始化输出矩阵 C（49 个 (16x16) 块堆叠）

ACC_GEMM = insn_GEMM(ACC, block_weight_matrix, INP_stack)
#assert(ACC_GEMM[0] == block_output_matrix[0][0])
print("ACC - Output matrix post-GEMM (", ACC_GEMM.shape[0], "x", ACC_GEMM.shape[1], ")")
print(ACC_GEMM)

## ALU

#### ReLU 激活（ReLU ACTIVATION）

ALU 操作：对 ACC 中的每个值，返回该值与 0 之间的最大值。目的是确保 ACC 中不存在负值。

In [ ]:
"""ReLU 激活"""

# 在 data_definitions/user_configuration.py 中，若 `useReLU=True`：

# ----------------------
# 定义 ALU-RELU UOP 缓冲区

if (len(uop_buffer) < 3 + 1):
    uop_buffer.append(structures_insn_uop.VTAUop( # UOP 3 - ALU (relu)
        dst_idx=0, 
        src_idx=0,
        wgt_idx=0
    ))

# ----------------------
# 定义 ALU-RELU 指令缓冲区

index_insn = 6 # 指令索引

if (len(insn_buffer) < index_insn + 1):
    insn_buffer.append(structures_insn_uop.VTAAluInsn( # I6: ALU - MAX IMM 0 (relu)
        opcode=4, # 4-ALU
        # 依赖标志
        pop_prev_dep=0,
        pop_next_dep=0,
        push_prev_dep=0,
        push_next_dep=0,
        # Operations
        reset=0, # 0-否, 1-重置
        uop_bgn=3, # UOP 3
        uop_end=4,
        loop_out=49,
        loop_in=16,
        # 未使用
        unused=0, # 未使用
        # 索引因子
        dst_factor_out=16,
        dst_factor_in=1, # ACC 每次递增 1
        src_factor_out=16,
        src_factor_in=1, # INP 每次递增 1
        alu_opcode=1, # 0-MIN, 1-MAX, 2-ADD, 3-SHR, 4-MUL
        use_imm=1, # 0-否, 1-是
        imm=0
    ))

# ----------------------
# 定义 ReLU 操作
def RELU(A, useReLU):
    if (useReLU):
        A = np.maximum(A, 0)
    return A

# ----------------------
# ALU ReLU 伪代码

def insn_RELU(ACC):
    for i0 in range(insn_buffer[index_insn].loop_in):
        for i1 in range(insn_buffer[index_insn].loop_out):
            for uop_index in range(insn_buffer[index_insn].uop_bgn, insn_buffer[index_insn].uop_end):
                X = uop_buffer[uop_index].dst_idx
                dst_idx = i0 * insn_buffer[index_insn].dst_factor_in + i1 * insn_buffer[index_insn].dst_factor_out + X # Index ACC
                ACC[dst_idx] = RELU(ACC[dst_idx], True) # 对 ACC 的每一行，对每个值执行 max(0, value)
    return ACC

# ----------------------
# 打印数据
# ----------------------

# 打印 ReLU UOP 缓冲区
print_uop_buffer("RELU", insn_buffer[index_insn].uop_bgn, insn_buffer[index_insn].uop_end)

# 打印 ReLU 指令缓冲区 
print_insn_buffer_ALU(index_insn, "RELU")

# 打印输出矩阵

ACC_ReLU = insn_RELU(ACC_GEMM)
print("\nACC - Output matrix post-ReLU (", ACC_ReLU.shape[0], "x", ACC_ReLU.shape[1], ")")
print(ACC_ReLU)

#### 平均池化（AVERAGE POOLING）
##### *第一次 ADD*

ALU 操作：将 ACC 中的两个向量相加，结果存入第一个向量的位置。

首先定义 UOP 缓冲区（指定数据读写位置），然后定义指令缓冲区。操作计算结果存入 ACC 并打印输出。

以下缓冲区对应的操作为：

>*LOOP 0 : ACC@(dst_idx=0) + ACC@(src_idx=1) => ACC@(dst_idx=0)*

>*LOOP 1 : ACC@(dst_idx=0 + 1 * dst_factor_in=2) + ACC@(src_idx=1 + 1 * src_factor_in=2) = ACC@2 + ACC@3 => ACC@2*

>*以此类推...*

In [ ]:
"""平均池化 - 第一次 ADD"""

# 此步骤后，相关数据存储量减半

# ----------------------
# 定义 ADD #1 UOP 缓冲区

if (len(uop_buffer) < 4 + 1):
    uop_buffer.append(structures_insn_uop.VTAUop( # UOP 4 - ALU (first add)
        dst_idx=0, 
        src_idx=1,
        wgt_idx=0
    ))

# ----------------------
# 定义 ADD #1 指令缓冲区

index_insn = 7 # 指令索引

if (len(insn_buffer) < index_insn + 1):
    insn_buffer.append(structures_insn_uop.VTAAluInsn( # I7: ALU - ADD (Average Pooling 1/3)
        opcode=4, # 4-ALU
        # 依赖标志
        pop_prev_dep=0,
        pop_next_dep=0,
        push_prev_dep=0,
        push_next_dep=0,
        # Operations
        reset=0, # 0-否, 1-重置
        uop_bgn=4, # UOP 4
        uop_end=5,
        loop_out=1,
        loop_in=392,
        # 未使用
        unused=0, # 未使用
        # 索引因子
        dst_factor_out=0,
        dst_factor_in=2, 
        src_factor_out=0,
        src_factor_in=2, 
        alu_opcode=2, # 0-MIN, 1-MAX, 2-ADD, 3-SHR, 4-MUL
        use_imm=0, # 0-否, 1-是
        imm=0
    ))

# ----------------------
# 定义 ADD 操作

def ADD(A, B):
    A = np.array(A)
    B = np.array(B)
    return A + B
        
# ----------------------
# ALU ADD 伪代码

def insn_ADD(ACC):
    for i0 in range(insn_buffer[index_insn].loop_in):
        for i1 in range(insn_buffer[index_insn].loop_out):
            for uop_index in range(insn_buffer[index_insn].uop_bgn, insn_buffer[index_insn].uop_end):
                X, Y = uop_buffer[uop_index].dst_idx, uop_buffer[uop_index].src_idx
                dst_idx = i0 * insn_buffer[index_insn].dst_factor_in + i1 * insn_buffer[index_insn].dst_factor_out + X
                inp_idx = i0 * insn_buffer[index_insn].src_factor_in + i1 * insn_buffer[index_insn].src_factor_out + Y
                ACC[dst_idx] = ADD(ACC[dst_idx], ACC[inp_idx])
    return ACC

# ----------------------
# 打印数据
# ----------------------

# 打印 ADD #1 UOP 缓冲区
print_uop_buffer("ADD #1", insn_buffer[index_insn].uop_bgn, insn_buffer[index_insn].uop_end)

# 打印 ADD #1 指令缓冲区 
print_insn_buffer_ALU(index_insn, "ADD #1")

# 打印输出矩阵
ACC_ADD1 = insn_ADD(ACC_ReLU)
print("\nACC - Output matrix post-first ADD (", ACC_ADD1.shape[0], "x", ACC_ADD1.shape[1], ")")
print(ACC_ADD1)

##### *第二次 ADD*

ALU 操作：将 ACC 中的两个向量相加，结果存入第一个向量的位置。

首先定义 UOP 缓冲区，然后定义指令缓冲区。操作计算结果存入 ACC 并打印输出。

以下缓冲区对应的操作为：

*LOOP_IN 0：*

>*LOOP_OUT 0 : ACC@(dst_idx=0) + ACC@(src_idx=28) => ACC@(dst_idx=0)*

>*LOOP_OUT 1 : ACC@(dst_idx=0 + 1 * dst_factor_out=56) + ACC@(src_idx=28 + 1 * src_factor_out=56) = ACC@56 + ACC@84 => ACC@56*

>*以此类推...*

*LOOP_IN 1：*

>*LOOP_OUT 0 : ACC@(dst_idx=0 + 1 * dst_factor_in=2) + ACC@(src_idx=28 + 1 * src_factor_in=2) = ACC@2 + ACC@30 => ACC@2*

>*LOOP_OUT 1 : ACC@(dst_idx=0 + 1 * dst_factor_out=56 + 1 * dst_factor_in=2) + ACC@(src_idx=28 + 1 * src_factor_out=56 + 1 * src_factor_in=2) = ACC@58 + ACC@84 => ACC@58*

*以此类推...*

In [ ]:
"""平均池化 - 第二次 ADD"""

# 此步骤后，相关数据存储量再次减半（共减少为原来的 1/4）

# ----------------------
# 定义 ADD #2 UOP 缓冲区

if (len(uop_buffer) < 5 + 1):
    uop_buffer.append(structures_insn_uop.VTAUop( # UOP 5 - ALU (second add)
        dst_idx=0, 
        src_idx=28,
        wgt_idx=0
    ))

# ----------------------
# 定义 ADD #2 指令缓冲区

index_insn = 8 # 指令索引

if (len(insn_buffer) < index_insn + 1):
    insn_buffer.append(structures_insn_uop.VTAAluInsn( # I8: ALU - ADD (Average Pooling 2/3)
        opcode=4, # 4-ALU
        # 依赖标志
        pop_prev_dep=0,
        pop_next_dep=0,
        push_prev_dep=0,
        push_next_dep=0,
        # Operations
        reset=0, # 0-否, 1-重置
        uop_bgn=5, # UOP 5
        uop_end=6,
        loop_out=14,
        loop_in=14,
        # 未使用
        unused=0, # 未使用
        # 索引因子
        dst_factor_out=56,
        dst_factor_in=2, 
        src_factor_out=56,
        src_factor_in=2, 
        alu_opcode=2, # 0-MIN, 1-MAX, 2-ADD, 3-SHR, 4-MUL
        use_imm=0, # 0-否, 1-是
        imm=0
    ))

# ----------------------
# 打印数据
# ----------------------

# 打印 ADD #2 UOP 缓冲区
print_uop_buffer("ADD #2", insn_buffer[index_insn].uop_bgn, insn_buffer[index_insn].uop_end)

# 打印 ADD #2 指令缓冲区 
print_insn_buffer_ALU(index_insn, "ADD #2")

# 打印输出矩阵
ACC_ADD2 = insn_ADD(ACC_ADD1)
print("\nACC - Output matrix post-second ADD (", ACC_ADD2.shape[0], "x", ACC_ADD2.shape[1], ")")
print(ACC_ADD2)

##### *右移（SHIFT-RIGHT）*

ALU 操作：对 ACC 向量中的每个值执行右移，即在本例中将值除以 2^2。

首先定义 UOP 缓冲区，然后定义指令缓冲区。操作计算结果存入 ACC 并打印输出。

以下缓冲区对应的操作为：

*LOOP_IN 0：*

>*LOOP_OUT 0 : ACC@(dst_idx=0) / 2^2 => ACC@(dst_idx=0)*

>*LOOP_OUT 1 : ACC@(dst_idx=0 + 1 * dst_factor_out=56) / 2^2 = ACC@56 / 2^2 => ACC@56*

>*以此类推...*

*LOOP_IN 1：*

>*LOOP_OUT 0 : ACC@(dst_idx=0 + 1 * dst_factor_in=2) / 2^2 = ACC@2 / 2^2 => ACC@2*

>*LOOP_OUT 1 : ACC@(dst_idx=0 + 1 * dst_factor_out=56 + 1 * dst_factor_in=2) / 2^2 = ACC@58 / 2^2 => ACC@58*

*以此类推...*

In [ ]:
"""平均池化 - SHR（右移）"""

# 此步骤对已累加的值求平均

# ----------------------
# 定义 SHR UOP 缓冲区

if (len(uop_buffer) < 6 + 1):
    uop_buffer.append(structures_insn_uop.VTAUop( # UOP 6 - ALU (shift right)
        dst_idx=0, 
        src_idx=0,
        wgt_idx=0
    ))

# ----------------------
# 定义 ALU-SHR 指令缓冲区

index_insn = 9 # 指令索引

if (len(insn_buffer) < index_insn + 1):
    insn_buffer.append(structures_insn_uop.VTAAluInsn( # I9: ALU - SHR (Average Pooling 3/3)
        opcode=4, # 4-ALU
        # 依赖标志
        pop_prev_dep=0,
        pop_next_dep=0,
        push_prev_dep=0,
        push_next_dep=1, # 发送就绪信号给 STORE
        # Operations
        reset=0, # 0-否, 1-重置
        uop_bgn=6, # UOP 6
        uop_end=7,
        loop_out=14,
        loop_in=14,
        # 未使用
        unused=0, # 未使用
        # 索引因子
        dst_factor_out=56,
        dst_factor_in=2, 
        src_factor_out=56,
        src_factor_in=2, 
        alu_opcode=3, # 0-MIN, 1-MAX, 2-ADD, 3-SHR, 4-MUL
        use_imm=1, # 0-否, 1-是
        imm=2 # 除以 4（向下取整）
    ))

# ----------------------
# 定义 SHR 操作

def SHR(A, IMM) :
    for i in range(len(A)): # A 由横向向量 (16x1) 组成
        A[i] = int(np.float64(A[i])) >> IMM
    return A

# ----------------------
# ALU SHR 伪代码

def insn_SHR(ACC):
    for i0 in range(insn_buffer[index_insn].loop_in):
        for i1 in range(insn_buffer[index_insn].loop_out):
            for uop_index in range(insn_buffer[index_insn].uop_bgn, insn_buffer[index_insn].uop_end):
                X = uop_buffer[uop_index].dst_idx
                dst_idx = i0 * insn_buffer[index_insn].dst_factor_in + i1 * insn_buffer[index_insn].dst_factor_out + X
                ACC[dst_idx] = SHR(ACC[dst_idx], insn_buffer[index_insn].imm)
    return ACC

# ----------------------
# 打印数据
# ----------------------

# 打印 SHR UOP 缓冲区
print_uop_buffer("SHR", insn_buffer[index_insn].uop_bgn, insn_buffer[index_insn].uop_end)

# 打印 SHR 指令缓冲区 
print_insn_buffer_ALU(index_insn, "SHR")

# 打印输出矩阵
ACC_SHR = insn_SHR(ACC_ADD2)
print("\nACC - Output matrix post-SHR (", ACC_SHR.shape[0], "x", ACC_SHR.shape[1], ")")
print(ACC_SHR)

##### 将数据存储到 DRAM（STORING THE DATA INTO DRAM）

计算完成后，本地存储的数据需要复制到 DRAM 中。以下缓冲区模拟 COMPUTE 模块与 STORE 模块之间的内存交互。

In [ ]:
"""从 SRAM 向 DRAM 存储数据"""

insn_buffer.append(structures_insn_uop.VTAMemInsn( # I10: STORE
    opcode=1, # 0-LOAD, 1-STORE, 3-FINISH
    # 依赖标志
    pop_prev_dep=1, # Acknowledge COMPUTE ready signal
    pop_next_dep=0,
    push_prev_dep=1, # 发送就绪信号给 COMPUTE
    push_next_dep=0,
    # 内存交互
    buffer_id=4, # 0-UOP, 1-WGT, 2-INP, 3-ACC, 4-OUT, 5-ACC8bit
    sram_base=0x0000,
    dram_base=0x00000300,
    unused=0, # 未使用
    # 数据操作
    y_size=1,
    x_size=784, # Store 49*16 OUT
    x_stride=784,
    y_pad_top=0,
    y_pad_bottom=0,
    x_pad_left=0,
    x_pad_right=0
))

insn_buffer.append(structures_insn_uop.VTAMemInsn( # I11: NOP-MEMORY-STAGE
    opcode=0, # 0-LOAD, 1-STORE, 3-FINISH
    # 依赖标志
    pop_prev_dep=0,
    pop_next_dep=0,
    push_prev_dep=0, 
    push_next_dep=1, # 发送就绪信号给 COMPUTE
    # 内存交互
    buffer_id=2, # 0-UOP, 1-WGT, 2-INP, 3-ACC, 4-OUT, 5-ACC8bit
    sram_base=0x0000,
    dram_base=0x00000000,
    unused=0, # 未使用
    # 数据操作
    y_size=0,
    x_size=0,
    x_stride=0,
    y_pad_top=0,
    y_pad_bottom=0,
    x_pad_left=0,
    x_pad_right=0
))

insn_buffer.append(structures_insn_uop.VTAMemInsn( # I12: NOP-COMPUTE-STAGE
    opcode=0, # 0-LOAD, 1-STORE, 3-FINISH
    # 依赖标志
    pop_prev_dep=1, # 确认 LOAD 就绪信号
    pop_next_dep=1, # 确认 STORE 就绪信号
    push_prev_dep=0,
    push_next_dep=0,
    # 内存交互
    buffer_id=0, # 0-UOP, 1-WGT, 2-INP, 3-ACC, 4-OUT, 5-ACC8bit
    sram_base=0x0000,
    dram_base=0x00000000,
    unused=0, # 未使用
    # 数据操作
    y_size=0,
    x_size=0,
    x_stride=0,
    y_pad_top=0,
    y_pad_bottom=0,
    x_pad_left=0,
    x_pad_right=0
))

insn_buffer.append(structures_insn_uop.VTAMemInsn( # I13: FINISH
    opcode=3, # 0-LOAD, 1-STORE, 3-FINISH
    # 依赖标志
    pop_prev_dep=0,
    pop_next_dep=0,
    push_prev_dep=0,
    push_next_dep=0,
    # 内存交互
    buffer_id=0, # 0-UOP, 1-WGT, 2-INP, 3-ACC, 4-OUT, 5-ACC8bit
    sram_base=0x0000,
    dram_base=0x00000000,
    unused=0, # 未使用
    # 数据操作
    y_size=0,
    x_size=0,
    x_stride=0,
    y_pad_top=0,
    y_pad_bottom=0,
    x_pad_left=0,
    x_pad_right=0
))

## 数据编码（DATA ENCODING）

**如何获取编码后的数据：**

每个 32 位的 UOP 和 128 位的指令都以十六进制对的形式编码。

*以 SHR 指令的编码为例：*

使用的结构体：

```
class VTAAluInsn(LittleEndianStructure):
    """ALU 指令结构（128 位）。"""
    _pack_ = 1
    _fields_ = [
        ("opcode", c_uint64, 3),
        ("pop_prev_dep", c_uint64, 1),
        ("pop_next_dep", c_uint64, 1),
        ("push_prev_dep", c_uint64, 1),
        ("push_next_dep", c_uint64, 1),
        ("reset", c_uint64, 1),
        ("uop_bgn", c_uint64, 13),
        ("uop_end", c_uint64, 14),
        ("loop_out", c_uint64, 14),
        ("loop_in", c_uint64, 14),
        ("unused", c_uint64, 1),
        ("dst_factor_out", c_uint64, 11),
        ("dst_factor_in", c_uint64, 11),
        ("src_factor_out", c_uint64, 11),
        ("src_factor_in", c_uint64, 11),
        ("alu_opcode", c_uint64, 3), # 0-MIN, 1-MAX, 2-ADD, 3-SHR, 4-MUL/SHL
        ("use_imm", c_uint64, 1), # 0-NO, 1-YES
        ("imm", c_uint64, 16)
    ]
```

缓冲区配置：

```
insn_buffer.append(structures_insn_uop.VTAAluInsn( # I9: ALU - SHR（平均池化 3/3）
        opcode=4,           # >>> 100 [3位]
        # DEP FLAG          # >>> 0001
        pop_prev_dep=0,
        pop_next_dep=0,
        push_prev_dep=0,
        push_next_dep=1,
        # 操作
        reset=0,            # >>> 0
        uop_bgn=6,          # >>> 0000 0000 0011 0
        uop_end=7,          # >>> 0000 0000 0001 11
        loop_out=14,        # >>> 0000 0000 0011 10
        loop_in=14,         # >>> 0000 0000 0011 10
        # 未使用
        unused=0,           # >>> 0
        # 索引因子
        dst_factor_out=56,  # >>> 0000 0111 000
        dst_factor_in=2,    # >>> 0000 0000 010
        src_factor_out=56,  # >>> 0000 0111 000
        src_factor_in=2,    # >>> 0000 0000 010
        alu_opcode=3,       # >>> 011
        use_imm=1,          # >>> 1
        imm=2               # >>> 0000 0000 0000 0010
    ))
```

将这个 128 位序列以小端格式（Little Endian）编码，在 .bin 文件中得到以下指令：**I9 : 44 06 e0 00 70 00 1c 00 38 10 00 0e 04 b0 02 00**

***二进制文件（BINARY FILES）***

要获取 `functional_simulator` 和 `cycle-accurate simulator` 所需的二进制文件，运行以下命令：

In [ ]:
%%capture
%run ../src/compiler/vta_compiler/operations_definition/examples/insn_lenet5_layer1.py

>> 运行以下命令访问文件中的数据：
- UOP 文件：`hexdump -C uop.bin > uop.txt`
- 指令文件：`hexdump -C instructions.bin > instructions.txt`

在 `compiler_output/` 文件夹中将得到：
- `uop.bin`：包含编码 UOP 的文件
- `instructions.bin`：包含编码指令的文件